# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [4]:
import os, subprocess, sys

# ── 1. Install uv ────────────────────────────────────────────────────────────
subprocess.run("wget -qO- https://astral.sh/uv/install.sh | sh", shell=True)
os.environ["PATH"] += os.pathsep + "/home/igadiya/.local/bin"

# ── 2. Create venv ───────────────────────────────────────────────────────────
subprocess.run("uv venv .venv --seed --clear", shell=True)
pip = ".venv/bin/python -m pip"

# ── 3. Install all dependencies in one shot ──────────────────────────────────
subprocess.run(f"""
{pip} install --quiet \
    "numpy<2" \
    "torch==2.5.1" \
    "transformers>=4.51,<4.57" \
    "tokenizers>=0.21,<0.22" \
    "accelerate>=0.30" \
    "peft>=0.10" \
    "bitsandbytes==0.43.3" \
    "vllm" \
    "nvidia-cuda-runtime-cu12" \
    "nvidia-cuda-cupti-cu12" \
    "nvidia-cudnn-cu12" \
    "pandas" \
    sympy tqdm sentencepiece ipykernel jupyter
""", shell=True, check=True)

# ── 4. Fix LD_LIBRARY_PATH for bitsandbytes CUDA 12 ─────────────────────────
venv_site = ".venv/lib/python3.12/site-packages"
ld_paths = [
    f"{venv_site}/nvidia/nvjitlink/lib",
    f"{venv_site}/nvidia/cuda_runtime/lib",
    "/usr/local/cuda/lib64",
    "/usr/local/cuda-12.8/lib64",
]
os.environ["LD_LIBRARY_PATH"] = ":".join(ld_paths) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

# Write it to .bashrc so it persists across sessions
bashrc_line = f'\nexport LD_LIBRARY_PATH={":".join(ld_paths)}:$LD_LIBRARY_PATH\n'
with open(os.path.expanduser("~/.bashrc"), "a") as f:
    f.write(bashrc_line)

# ── 5. Register Jupyter kernel ───────────────────────────────────────────────
subprocess.run(
    ".venv/bin/python -m ipykernel install --user --name cse151bV2 --display-name 'Python (cse151bV2)'",
    shell=True
)

print("\n✅ Done. Select kernel: top right → 'Python (cse151bV2)' → restart.")

downloading uv 0.11.8 x86_64-unknown-linux-gnu


installing to /home/igadiya/.local/bin
  uv
  uvx
everything's installed!


Using CPython 3.11.9 interpreter at: /opt/conda/bin/python
Creating virtual environment with seed packages at: .venv
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + packaging==26.2
 + pip==26.1
 + setuptools==82.0.1
 + wheel==0.47.0
Activate with: source .venv/bin/activate


Installed kernelspec cse151bV2 in /home/igadiya/.local/share/jupyter/kernels/cse151bv2

✅ Done. Select kernel: top right → 'Python (cse151bV2)' → restart.


In [3]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 4096

DEVICE = "cuda"

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [2]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: 'data/public.jsonl'

In [3]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
)

train_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B-Thinking-2507",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [16]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Step A: Prepare the model for 4-bit training
# This handles technical details like layer-norm precision for quantized models
train_model = prepare_model_for_kbit_training(train_model)

# Step B: Configure LoRA
lora_config = LoraConfig(
    r=16,                # Rank: Higher r means more trainable parameters
    lora_alpha=32,       # Scaling factor (usually 2x Rank)
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Step C: Wrap the model
train_model = get_peft_model(train_model, lora_config)

# Step D: Verify (This should show only ~1-2% of params are trainable)
train_model.print_trainable_parameters()

trainable params: 11,796,480 || all params: 4,034,264,576 || trainable%: 0.2924


In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
    padding_side="left",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded:", tokenizer.__class__.__name__)

Tokenizer loaded: Qwen2Tokenizer


In [18]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning.\n\n "
    "MANDATORY RULE:\n"
    "- The FIRST token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the first token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{final_answer}\n"
    "Brief justification (<= 2 sentences).\n\n "
    "CONSTRAINTS:\n"
    "- Do NOT think before answering.\n"
    "- Do NOT delay the boxed answer.\n"
    "- Output EXACTLY 2 lines.\n"
    "- No extra text.\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician reasoning.\n\n"
    "MANDATORY RULE:\n"
    "- The FIRST token MUST be: \\boxed{X} where X is the answer choice.\n "
    "- If this is not the first token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{X}\n"
    "Brief justification (<= 2 sentences).\n\n"
    "CONSTRAINTS:\n"
    "- Do NOT think before answering.\n"
    "- Do NOT delay the boxed answer.\n"
    "- Output EXACTLY 2 lines.\n"
    "- No extra text.\n"
)

SYSTEM_PROMPT_REFLECTION_FREE = (
    "Double check the provided answer and provide the correct answer to the question."
    "MANDATORY RULE:"
    "- Final answer MUST be: \\boxed{X} where X = correct answer."
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_REFLECTION_MCQ = (
    "Double check the provided answer and provide the correct answer to the question."
    "MANDATORY RULE:\n"
    "- Final answer MUST be: \\boxed{X} where X = correct option."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def build_reflection_prompt(question: str, options: Optional[list]) -> tuple[str, str, str]:
    """Return (system_prompt, user_prompt, reflection_prompt) for a question."""
    system_prompt, user_prompt = build_prompt(question, options)
    if options:
        return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_MCQ
    return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_FREE

# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

NameError: name 'mcq_sample' is not defined

In [19]:
from torch.utils.data import Dataset

class MathDataset(Dataset):
    """Tokenizes question→answer pairs for supervised fine-tuning."""

    def __init__(self, data, tokenizer, max_length=512):
        self.samples = []
        for item in data:
            system, user = build_prompt(item["question"], item.get("options"))
            answer = item["answer"]
            if isinstance(answer, list):
                answer = ", ".join(str(a) for a in answer)

            messages = [
                {"role": "system",    "content": system},
                {"role": "user",      "content": user},
                {"role": "assistant", "content": f"\\boxed{{{answer}}}"},
            ]
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            tokenized = tokenizer(
                text,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt",
            )
            input_ids = tokenized["input_ids"].squeeze()
            labels = input_ids.clone()

            # Mask prompt tokens — only compute loss on the answer
            assistant_token = tokenizer.encode(
                "<|im_start|>assistant\n", add_special_tokens=False
            )
            for i in range(len(labels) - len(assistant_token)):
                if labels[i : i + len(assistant_token)].tolist() == assistant_token:
                    labels[:i + len(assistant_token)] = -100
                    break

            self.samples.append({"input_ids": input_ids, "labels": labels})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


train_dataset = MathDataset(data, tokenizer, max_length=512)
print(f"Training samples: {len(train_dataset)}")

Training samples: 1126


In [23]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="data/lora_math_adapter",
    num_train_epochs=2,
    per_device_train_batch_size=2,       # keep low for 4-bit on Colab
    gradient_accumulation_steps=8,       # effective batch = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",            # memory-efficient for 4-bit
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=train_model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Trainer(
    model=train_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

trainer.train()

/home/igadiya/private/151BMathematicalReasoning/.venv/lib/python3.11/site-packages/transformers/data/data_collator.py:741: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/igadiya/private/151BMathematicalReasoning/.venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences 

Step,Training Loss
10,2.155600
20,0.954700
30,0.789800
40,0.631600
50,0.559700
60,0.597000
70,0.594700
80,0.533800
90,0.518400
100,0.582600


/home/igadiya/private/151BMathematicalReasoning/.venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=142, training_loss=0.7080259658920933, metrics={'train_runtime': 888.4686, 'train_samples_per_second': 2.535, 'train_steps_per_second': 0.16, 'total_flos': 2.5218768546299904e+16, 'train_loss': 0.7080259658920933, 'epoch': 2.0})

In [4]:
# Save only the small LoRA adapter weights (~tens of MB)
ADAPTER_PATH = "data/lora_math_adapter/final_adapter"
train_model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"Adapter saved to {ADAPTER_PATH}")

NameError: name 'train_model' is not defined

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

## 5. Load Model with vLLM

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [ ]:
import torch
print(torch.device("cuda"))

cuda


In [1]:
import gc

# Release train_model (LoRA-wrapped model used during fine-tuning)
try:
    train_model.cpu()
    del train_model
    print("train_model deleted")
except NameError:
    pass

# Release HF model if it was loaded separately
try:
    model.cpu()
    del model
    print("model deleted")
except NameError:
    pass

# Release tokenizer
try:
    del tokenizer
    print("tokenizer deleted")
except NameError:
    pass

gc.collect()

import torch
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory freed. Allocated: 0.00 GB


In [4]:
DATA_PATH = "data/private.jsonl"
data = [json.loads(line) for line in open(DATA_PATH)]
print(f"Loaded {len(data)} examples")

Loaded 943 examples


In [5]:
ADAPTER_PATH = "data/lora_math_adapter/final_adapter"

In [6]:
import subprocess
subprocess.run(".venv/bin/pip install -q 'peft>=0.14.0' 'transformers>=4.51.0'", shell=True)

CompletedProcess(args=".venv/bin/pip install -q 'peft>=0.14.0' 'transformers>=4.51.0'", returncode=0)

In [7]:
import subprocess
subprocess.run(".venv/bin/pip install -q 'vllm>=0.4.3'", shell=True)

CompletedProcess(args=".venv/bin/pip install -q 'vllm>=0.4.3'", returncode=0)

In [7]:
import psutil
print(f"Available RAM: {psutil.virtual_memory().available/1e9:.1f} GB")
print(f"Total RAM: {psutil.virtual_memory().total/1e9:.1f} GB")

Available RAM: 508.7 GB
Total RAM: 540.6 GB


In [8]:
import torch
from transformers import AutoModelForCausalLM
import os

# Find where HuggingFace cached the model during training
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
print(os.listdir(cache_dir))

['version.txt', '.locks', 'models--Qwen--Qwen3-4B-Thinking-2507']


In [1]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

GPU free: 25.0 GB


In [6]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MERGED_PATH = "/tmp/lora_math_merged"

print("1. Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="cuda",
    local_files_only=True,
)
print("2. Loading adapter...")
model_merged = PeftModel.from_pretrained(base, ADAPTER_PATH)
print("3. Merging...")
model_merged = model_merged.merge_and_unload()
print("4. Saving...")
model_merged.save_pretrained(MERGED_PATH)
AutoTokenizer.from_pretrained(ADAPTER_PATH).save_pretrained(MERGED_PATH)
print("Done.")

1. Loading base model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

2. Loading adapter...
3. Merging...
4. Saving...
Done.


In [7]:
import gc, torch

del base
del model_merged
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

GPU free: 24.9 GB


In [8]:
from vllm import LLM, SamplingParams
llm = LLM(
    model=MERGED_PATH,
    dtype="half",
    gpu_memory_utilization=0.85,
    max_model_len=4096,
    trust_remote_code=True,
    max_num_seqs=32,
    # no enable_lora
)
# No lora_request needed — pass None or remove from generate calls
lora_request = None

INFO 04-30 17:58:56 __init__.py:207] Automatically detected platform cuda.
INFO 04-30 17:59:08 config.py:549] This model supports multiple tasks: {'embed', 'reward', 'generate', 'classify', 'score'}. Defaulting to 'generate'.
INFO 04-30 17:59:08 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='/tmp/lora_math_merged', speculative_config=None, tokenizer='/tmp/lora_math_merged', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=Fal

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 04-30 17:59:14 model_runner.py:1115] Loading model weights took 7.5454 GB
INFO 04-30 17:59:16 worker.py:267] Memory profiling takes 2.13 seconds
INFO 04-30 17:59:16 worker.py:267] the current vLLM instance can use total_gpu_memory (23.50GiB) x gpu_memory_utilization (0.85) = 19.97GiB
INFO 04-30 17:59:16 worker.py:267] model weights take 7.55GiB; non_torch_memory takes 0.01GiB; PyTorch activation peak memory takes 0.30GiB; the rest of the memory reserved for KV Cache is 12.12GiB.
INFO 04-30 17:59:16 executor_base.py:111] # cuda blocks: 5514, # CPU blocks: 1820
INFO 04-30 17:59:16 executor_base.py:116] Maximum concurrency for 4096 tokens per request: 21.54x
INFO 04-30 17:59:27 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_uti

Capturing CUDA graph shapes: 100%|██████████| 7/7 [00:03<00:00,  1.96it/s]

INFO 04-30 17:59:31 model_runner.py:1562] Graph capturing finished in 4 secs, took 0.98 GiB
INFO 04-30 17:59:31 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 17.04 seconds


In [9]:
sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

In [10]:
def generate_batch(prompts, max_new_tokens):
    outputs = llm.generate(prompts, sampling_params)
    return [o.outputs[0].text.strip() for o in outputs]

## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [11]:
def extract_boxed(text):
    m = re.search(r"\\boxed\{.*?\}", text)
    if m:
        return m.group(0)
    return text[:200]

In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH, trust_remote_code=True)
temp_output_file = "data/responses_checkpoint.csv"

In [13]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician reasoning.\n\n "
    "MANDATORY RULE:\n"
    "- The FIRST token MUST be: \\boxed{final_answer}\n "
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}.\n "
    "- If this is not the first token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{final_answer}\n"
    "Brief justification (<= 2 sentences).\n\n "
    "CONSTRAINTS:\n"
    "- Do NOT think before answering.\n"
    "- Do NOT delay the boxed answer.\n"
    "- Output EXACTLY 2 lines.\n"
    "- No extra text.\n"
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician reasoning.\n\n"
    "MANDATORY RULE:\n"
    "- The FIRST token MUST be: \\boxed{X} where X is the answer choice.\n "
    "- If this is not the first token, the response is invalid.\n\n"
    "FORMAT:\n"
    "\\boxed{X}\n"
    "Brief justification (<= 2 sentences).\n\n"
    "CONSTRAINTS:\n"
    "- Do NOT think before answering.\n"
    "- Do NOT delay the boxed answer.\n"
    "- Output EXACTLY 2 lines.\n"
    "- No extra text.\n"
)

SYSTEM_PROMPT_REFLECTION_FREE = (
    "Double check the provided answer and provide the correct answer to the question."
    "MANDATORY RULE:"
    "- Final answer MUST be: \\boxed{X} where X = correct answer."
    "- If multiple sub-answers are required, put them in one box separated by commas, e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_REFLECTION_MCQ = (
    "Double check the provided answer and provide the correct answer to the question."
    "MANDATORY RULE:\n"
    "- Final answer MUST be: \\boxed{X} where X = correct option."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

def build_reflection_prompt(question: str, options: Optional[list]) -> tuple[str, str, str]:
    """Return (system_prompt, user_prompt, reflection_prompt) for a question."""
    system_prompt, user_prompt = build_prompt(question, options)
    if options:
        return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_MCQ
    return system_prompt, user_prompt, SYSTEM_PROMPT_REFLECTION_FREE

In [14]:
import pandas as pd
import re

# 1. Load existing progress if it exists
if os.path.exists(temp_output_file):
    df_existing = pd.read_csv(temp_output_file)
    completed_ids = set(df_existing['id'].tolist())
    print(f"Resuming from checkpoint. {len(completed_ids)} items already processed.")
else:
    completed_ids = set()
    pd.DataFrame(
        columns=["id", "question", "initial_response", "revised_response", "has_box"]
    ).to_csv(temp_output_file, index=False)

# Filter remaining data
remaining_data = [item for item in data if item.get("id") not in completed_ids]

print(f"Processing {len(remaining_data)} remaining questions...")

batch_size = 8

for i in range(0, len(remaining_data), batch_size):
    items = remaining_data[i : i + batch_size]

    # ── Build initial prompts ─────────────────────────────
    prompts = []
    for item in items:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    all_responses = generate_batch(prompts, MAX_TOKENS)


    revised_prompts = []
    for j, item in enumerate(items):
        system, user, reflection = build_reflection_prompt(
            item["question"], item.get("options")
        )

        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
                {"role": "assistant", "content": extract_boxed(all_responses[j])},
                {"role": "user", "content": reflection},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        revised_prompts.append(prompt_text)

    # ── Reflection generation ────────────────────────────
    all_revised_responses = generate_batch(
        revised_prompts, MAX_TOKENS
    )

    # ── Collect results ──────────────────────────────────
    results = []
    for j, item in enumerate(items):
        revised = all_revised_responses[j]

        results.append({
            "id": item.get("id"),
            "question": item.get("question"),
            "initial_response": all_responses[j],
            "revised_response": revised,
            "has_box": bool(re.search(r"\\boxed\{.*?\}", revised)),
        })

    # ── Save checkpoint ──────────────────────────────────
    df_batch = pd.DataFrame(results)
    df_batch.to_csv(temp_output_file, mode="a", index=False, header=False)

    # ── Debug print ──────────────────────────────────────
    for k, item in enumerate(items):
        print(f"\n── Response {k} (id={item.get('id')}) ──")
        print(all_revised_responses[k][:200])  # preview

Resuming from checkpoint. 800 items already processed.
Processing 143 remaining questions...


Processed prompts: 100%|██████████| 8/8 [01:05<00:00,  8.15s/it, est. speed input: 66.68 toks/s, output: 76.79 toks/s]  



── Response 0 (id=800) ──
<think>

</think>

\boxed{120}

── Response 1 (id=801) ──
This is the second thought. I need to carefully re-evaluate the problem.

The problem asks for the dimension of $\Psi_k = \{\phi: V \to W : \, \mathrm{rank} (\phi|_A) \leq k \}$ as a subvariety of $\m

── Response 2 (id=802) ──
<think>

</think>

\boxed{26/7}

── Response 3 (id=803) ──
<think>

</think>

\boxed{22}

── Response 4 (id=804) ──
<think>

</think>

\boxed{H}

── Response 5 (id=805) ──
The problem is to determine the corresponding output sequence y_list given the input x_list [20, 21, 22, 23, 24, 25, 26, 27, 28, 29].

I am considering the options provided:

A. [40386519, 96018828, 2

── Response 6 (id=806) ──
<think>

</think>

\boxed{-34/45, -34/45, 45/34, 34/45, -45/34, -45/34}

── Response 7 (id=807) ──
<think>

</think>

\boxed{J}


Processed prompts: 100%|██████████| 8/8 [01:14<00:00,  9.33s/it, est. speed input: 50.03 toks/s, output: 193.67 toks/s] 



── Response 0 (id=808) ──
The question asks for the output sequence y_list corresponding to the input x_list = [57, 58, 59, 60, 61, 62, 63, 64, 65, 66].

Let's verify the answer choices:

A. [24, 26, 27, 20, 21, 23, 22, 65, 64

── Response 1 (id=809) ──
<think>

The problem involves determining the minimum number of roads $S(n)$ in ELMOpia with $n \ge 4$ cities, where for any two cities $A$ and $B$, there exist distinct cities $C$ and $D$ such that $

── Response 2 (id=810) ──
The question asks for the general solution of the differential equation $y'' + y = \sin x$.

Let's think about this carefully.

This is a nonhomogeneous linear differential equation. The general solut

── Response 3 (id=811) ──
<think>

</think>

\boxed{1.336E-19}

── Response 4 (id=812) ──
<think>

</think>

\boxed{8*r^(12)}

── Response 5 (id=813) ──
<think>

</think>

\boxed{A, C}

── Response 6 (id=814) ──
<think>

</think>

\boxed{5, 1.95, 0.39, 1.84, 31, 1.89, 0.06, 36, 2.99, 0.06}

── Response 7 (id=815) 

...


── Response 0 (id=920) ──
<think>

\boxed{187}

── Response 1 (id=921) ──
<think>

</think>

\boxed{C}

── Response 2 (id=922) ──
The question is to evaluate the contour integral $\oint_{c} \frac{2 z^{2}-z+1} {z-1} \mathrm{d} z$ where $C$ is the circle $|z|=2$.

First, I'll identify the singularities of the integrand. The integr

── Response 3 (id=923) ──
<think>

</think>

\boxed{y=-6.2*x+334.2, 222.2, 53.9}

── Response 4 (id=924) ──
The problem asks: According to one mathematical model, the average life expectancy for American men born in 1900 was 55 years. Life expectancy has increased by about 0.2 year for each birth year after

── Response 5 (id=925) ──
<think>

</think>

\boxed{80}

── Response 6 (id=926) ──
<think>

</think>

\boxed{-3*t^2-6*t+2}

── Response 7 (id=927) ──
<think>

</think>

\boxed{false, true, true, true, false, false}


Processed prompts: 100%|██████████| 8/8 [01:04<00:00,  8.08s/it, est. speed input: 59.65 toks/s, output: 128.16 toks/s]



── Response 0 (id=928) ──
This is an expert reasoning question. I need to determine the correct cosine series representation of the given piecewise function f(x).

The function is defined as:
f(x) = 0 for 0 ≤ x < π/2
f(x) = 1 

── Response 1 (id=929) ──
This is an exercise in calculating a pooled variance and constructing a confidence interval for the difference in means between two independent groups, followed by a hypothesis test conclusion.

Let's

── Response 2 (id=930) ──
This is a reasoning task. I need to determine the correct answer choice by analyzing the determinant and the cofactors.

First, let me understand what's being asked: We need to find the sum of the cof

── Response 3 (id=931) ──
<think>

</think>

\boxed{3, -3}

── Response 4 (id=932) ──
<think>

</think>

\boxed{26}

── Response 5 (id=933) ──
<think>

</think>

\boxed{C}

── Response 6 (id=934) ──
<think>

The task is to determine the corresponding output sequence y_list given the input x_list = [64, 65, 66, 67,

Processed prompts: 100%|██████████| 7/7 [01:10<00:00, 10.06s/it, est. speed input: 38.41 toks/s, output: 104.46 toks/s] 


── Response 0 (id=936) ──
<think>

</think>

\boxed{-23}

── Response 1 (id=937) ──
<think>

</think>

\boxed{320}

── Response 2 (id=938) ──
<think>

</think>

\boxed{49.88}

── Response 3 (id=939) ──
The problem asks for the sum S of all positive integers k less than 100 that are relatively prime to 100. Then, it wants the floor of S divided by 10^(n-2) where 10^n <= S < 10^(n+1).

First, I need t

── Response 4 (id=940) ──
This is the second instance of the problem. I need to provide the final answer choice.

Let me analyze the sequence definition and compute the values:

The definition: a(n) is the least gap in the par

── Response 5 (id=941) ──
<think>

</think>

\boxed{A, C, D, A, B, C}

── Response 6 (id=942) ──
<think>

</think>

\boxed{16, 1}


## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [2]:
import subprocess
subprocess.run(".venv/bin/pip install -q 'antlr4-python3-runtime==4.11.1'", shell=True)

CompletedProcess(args=".venv/bin/pip install -q 'antlr4-python3-runtime==4.11.1'", returncode=0)

In [8]:
#for submission

import pandas as pd

df = pd.read_csv("data/responses_checkpoint.csv")

submission = df[["id", "revised_response"]].rename(columns={"revised_response": "response"})
submission.to_csv("data/submission.csv", index=False)
print(f"Saved {len(submission)} rows to data/submission.csv")
print(submission.head(3))

Saved 943 rows to data/submission.csv
   id                                        response
0   0            <think>\n\n</think>\n\n\boxed{3, 16}
1   1                <think>\n\n</think>\n\n\boxed{C}
2   2  <think>\n\n</think>\n\n\boxed{2.133, 1.645, B}


In [7]:
import pandas as pd, re, sys
from tqdm import tqdm

# Load all responses from checkpoint
df = pd.read_csv("data/responses_checkpoint.csv")
print(f"Loaded {len(df)} responses from checkpoint")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", str(text).upper())
    return matches[-1] if matches else ""

def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()

sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

# Build a lookup from id -> item for gold answers
data_lookup = {item["id"]: item for item in data}

results = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Scoring"):
    item = data_lookup.get(row["id"])
    if item is None:
        continue
    response = row["revised_response"]
    is_mcq = bool(item.get("options"))
    gold = item["answer"]

    if is_mcq:
        correct = score_mcq(str(response), str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":      row["id"],
        "is_mcq":  is_mcq,
        "gold":    gold,
        "response": response,
        "correct": correct,
    })

print(f"Scoring complete. {len(results)} results.")

Loaded 943 responses from checkpoint


Scoring:   0%|          | 0/943 [00:00<?, ?it/s]


KeyError: 'answer'

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!